In [ ]:
# Load the dataset that is given to you 
# Import libraries

df = pd.read_csv("home_loan_data.csv")
df.head()

In [ ]:
# Check for null values in the dataset 
null_percent = (df.isnull().sum())/100

# handle Missing value
df.fillna(df.median(numeric_only=True), inplace=True) 
df.fillna(df.mode.iloc[0], inplace=True)

In [ ]:
# Print the percentage of default to a payer of the dataset for the TARGET column
target_counts = df['TARGET'].value_counts(normalize=True)*100
print - target_counts

sns.countplot(x='TARGET', data=df)
plt.title("Default vs Non Default")
plt.show()


In [ ]:
# Balance the dataset if the data is imbalanced 
# SMOTE

X = df.drop('TARGET', axis=1)
y = df['TARGET']

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

sns.countplot(x=y_resampled)
plt.title("Balanced Target Distribution")
plt.show()

In [ ]:
# Encode the columns that are required for the model

cat_cols = X_resampled.select_dtype(include='object').columns
le = LabelEncoder()
for col in cat_cols :
    X_resampled[col] = le.fit_transform(X_resampled[col])

# Feature Scaling
scaler = StandardScaler
X_sclaed = scaler.fit_transform(X_resampled)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
         X_scaled, y_resampled, test_size=0.2, random_state=42)

# Build DL Model
model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
      optimizer='adam',
      loss='binary_crossentropy',
      metrics = [
                Recall(name='sensitivity')
      ]
)

# Model Training
history = model.fit(
          X_train, y_train,
          validation_split = 0.2,
          epochs = 10,
          batch_size = 256,
          verbose=1
)

# Model Evaluation
results = model.evaluate(X_test, y_test, verbose=0)
print -> evaluation results

# Predictions
y_pred -> model.predict -> 